<a href="https://colab.research.google.com/github/rxbass/Fine-Tuning-Jo/blob/main/FineTuneFirstLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install transformers datasets peft trl accelerate
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
print("Model and tokenizer loaded.")



/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model and tokenizer loaded.


In [2]:
from datasets import Dataset

data = [
    {"messages": [
        {"role": "user", "content": "How long does delivery take?"},
        {"role": "assistant", "content": "Hi! GreenCart delivers within 2-3 business days."},
    ]},
    {"messages": [
        {"role": "user", "content": "Can I return an item?"},
        {"role": "assistant", "content": "Of course! You can return any item within 7 days for a full refund."},
    ]},
    {"messages": [
        {"role": "user", "content": "Do you deliver on Sundays?"},
        {"role": "assistant", "content": "We deliver Monday to Saturday. Sunday delivery is coming soon!"},
    ]},
]
dataset = Dataset.from_list(data)
print(dataset)

Dataset({
    features: ['messages'],
    num_rows: 3
})


In [3]:
from peft import LoraConfig

lora_config = lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

In [4]:
pip install --upgrade torchao

In [5]:
%pip install -U trl

In [6]:
# !pip install trl
import torch
from trl import SFTTrainer, SFTConfig

has_gpu = torch.cuda.is_available()

training_args = SFTConfig(
    output_dir="finetuned_model",
    num_train_epochs=3,             # go over the data 3 times
    per_device_train_batch_size=1,
    learning_rate=2e-4,
    logging_steps=1,                # print the loss every step
    report_to="none",               # no external logging
    use_cpu=not has_gpu,            # train on CPU when there is no GPU (e.g. a laptop)
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,        # <-- makes it LoRA fine-tuning
)

trainer.train()   # watch the loss go DOWN
print("Training done!")

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Step,Training Loss
1,4.163009
2,4.092124
3,3.372813
4,3.368636
5,4.051661
6,3.975577
7,3.282331
8,3.950990
9,3.980809


Training done!
